In [46]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import mesa
import pandas as pd

from mesa.visualization import SolaraViz, make_plot_component, make_space_component


In [54]:
def update_expectation(expected_outcome, outcome, alpha):
    expected_outcome += alpha * (outcome - expected_outcome)
    return expected_outcome



class AdoptionAgent(mesa.Agent):
    """An agent with skill, and the ability to use collective intelligence."""

    def __init__(self, model):
        """initialize an AdoptionAgent instance.

        Args:
            model: A model instance
        """
        super().__init__(model)
        
        "Preset starting variables for the model:"
        self.skill = np.clip(np.random.normal(loc=0.5, scale=0.2), 0, 1)
        self.ability = 0
        self.wins = 1
        self.losses = 1
        self.aichoice = 0
        self.expected = 0.5
        self.outcome = 0
        self.luck = 0

        "Variables that can be changed between batches:"
        self.alpha = 0.1
        self.beta = 0.05
        self.boost = 0.10
        self.middle = 0.10
        self.scaler = 0.02
        


    def move(self):
        """move to a random neighboring cell."""
        possible_steps = self.model.grid.get_neighborhood(
            self.pos, moore=True, include_center=False
        )
        new_position = self.random.choice(possible_steps)
        self.model.grid.move_agent(self, new_position)

    def compete(self):
        """One agent picks another agent randomly and compares scores updating all info"""
        cellmates = self.model.grid.get_cell_list_contents([self.pos])
        if len(cellmates) > 1:
            other = self.random.choice(cellmates)


            self.aichoice = np.random.choice([0, 1], p=[1 - self.expected, self.expected])
            other.aichoice = np.random.choice([0, 1], p=[1 - other.expected, other.expected])

            self.luck = np.clip(np.random.normal(loc= middle , scale= scaler ), 0, 1)
            other.luck = np.clip(np.random.normal(loc= middle , scale= scaler ), 0, 1)

            self.ability = self.skill + self.aichoice * self.boost + self.luck
            other.ability = other.skill + other.aichoice * boost + other.luck
            self.skill = np.clip(self.skill + ((-1)**self.aichoice ) * self.beta, 0, 1)
            if self.ability > other.ability:
                self.wins += 1
                self.outcome = 1 * self.aichoice
                self.expected = update_expectation(self.expected, self.outcome, self.alpha)

            else:
                self.losses += 1
                self.outcome = 1 - self.aichoice
                self.expected = update_expectation(self.expected, self.outcome, self.alpha)




    def step(self):
        """do one step of the agent."""
        self.move()
        self.compete()


class AdoptionModel(mesa.Model):
    """A model with some number of agents."""

    def __init__(self,
                 alpha = 0.1,
                 beta = 0.05,
                 boost = 0.10,
                 middle = 0.10,
                 scaler = 0.02,
                 n=10,
                 width=10,
                 height=10,
                 seed=None):
        
        """Initialize a competition instance.

        Args:
            N: The number of agents.
            width: width of the grid.
            height: Height of the grid.
        """
        super().__init__(seed=seed)
        self.num_agents = n
        self.alpha = alpha
        self.beta = beta
        self.boost = boost
        self.middle = middle
        self.scaler = scaler
        self.grid = mesa.space.MultiGrid(width, height, True)

        # Create agents
        agents = AdoptionAgent.create_agents(model=self, n=n)
        # Create x and y positions for agents
        x = self.rng.integers(0, self.grid.width, size=(n,))
        y = self.rng.integers(0, self.grid.height, size=(n,))
        for a, i, j in zip(agents, x, y):
            # Add the agent to a random grid cell
            self.grid.place_agent(a, (i, j))

        self.datacollector = mesa.DataCollector(
            agent_reporters={
                "Skill": "skill",
                "Ability": "ability",
                "Wins": "wins",
                "Losses": "losses",
                "AI Usage Choice": "aichoice",
                "Expected": "expected",
                "Outcome": "outcome",
                "Luck" : "luck"
            }
        )
        self.datacollector.collect(self)

    def step(self):
        """do one step of the model"""
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)

In [55]:
model = AdoptionModel(100)
for _ in range(200):
    model.step()


data = model.datacollector.get_agent_vars_dataframe()
# Use seaborn

data


Skill   Ability  Wins  Losses  AI Usage Choice  Expected  \
Step AgentID                                                                
0    1        0.685222  0.000000     1       1                0  0.500000   
     2        0.235681  0.000000     1       1                0  0.500000   
     3        0.309050  0.000000     1       1                0  0.500000   
     4        0.321251  0.000000     1       1                0  0.500000   
     5        0.614028  0.000000     1       1                0  0.500000   
...                ...       ...   ...     ...              ...       ...   
200  6        0.847465  1.028734    19      10                1  0.543170   
     7        0.100000  0.303967     2      25                1  0.477792   
     8        0.066122  0.287263     4       7                1  0.462390   
     9        0.050000  0.266932     4      20                1  0.494957   
     10       0.307113  0.386572     6      17                0  0.573652   

              Outcome      Luck  
Step AgentID                     
0    1              0  0.000000  
     2              0  0.000000  
     3              0  0.000000  
     4              0  0.000000  
     5              0  0.000000  
...               ...       ...  
200  6              0  0.081269  
     7              0  0.053967  
     8              0  0.071142  
     9              0  0.066932  
     10             1  0.079459  

[2010 rows x 8 columns]

In [56]:
params = {"width": 10,
          "height": 10,
          "alpha": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
          "beta": 0.05,
          "boost": 0.05,
          "middle": 0.05,
          "scaler": 0.05,
          }

results = mesa.batch_run(
    AdoptionModel,
    parameters=params,
    iterations=5,
    max_steps=100,
    number_processes=1,
    data_collection_period=1,
    display_progress=True,
)


results_df = pd.DataFrame(results)
print(results_df.keys())

  0%|          | 0/30 [00:00<?, ?it/s]

Index(['RunId', 'iteration', 'Step', 'width', 'height', 'alpha', 'beta',
       'boost', 'middle', 'scaler', 'AgentID', 'Skill', 'Ability', 'Wins',
       'Losses', 'AI Usage Choice', 'Expected', 'Outcome', 'Luck'],
      dtype='object')


In [57]:
results_df

,RunId,iteration,Step,width,height,alpha,beta,boost,middle,scaler,AgentID,Skill,Ability,Wins,Losses,AI Usage Choice,Expected,Outcome,Luck
0,0,0,0,10,10,0.0,0.05,0.05,0.05,0.05,1,0.360151,0.000000,1,1,0,0.500000,0,0.000000
1,0,0,0,10,10,0.0,0.05,0.05,0.05,0.05,2,0.450116,0.000000,1,1,0,0.500000,0,0.000000
2,0,0,0,10,10,0.0,0.05,0.05,0.05,0.05,3,0.637993,0.000000,1,1,0,0.500000,0,0.000000
3,0,0,0,10,10,0.0,0.05,0.05,0.05,0.05,4,0.353350,0.000000,1,1,0,0.500000,0,0.000000
4,0,0,0,10,10,0.0,0.05,0.05,0.05,0.05,5,0.115730,0.000000,1,1,0,0.500000,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30295,29,4,100,10,10,0.5,0.05,0.05,0.05,0.05,6,0.573177,0.658162,7,5,0,0.523811,1,0.084985
30296,29,4,100,10,10,0.5,0.05,0.05,0.05,0.05,7,0.215678,0.493828,1,6,1,0.449145,0,0.128149
30297,29,4,100,10,10,0.5,0.05,0.05,0.05,0.05,8,0.574738,0.585968,3,7,0,0.521866,0,0.061229
30298,29,4,100,10,10,0.5,0.05,0.05,0.05,0.05,9,0.084078,0.172446,2,13,0,0.435524,0,0.088367
